In [1]:
from datetime import datetime, timedelta
import json
import uuid
import random 
from sqlalchemy import create_engine

from utils import reset_db, get_session, model_to_dict
from data.models import cultpass

# Udahub Accounts

## Cultpass Database

**Init DB**

In [2]:
cultpass_db = "data/external/cultpass.db"

In [3]:
reset_db(cultpass_db)

✅ Removed existing data/external/cultpass.db
2026-08-16 19:56:38,380 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-08-16 19:56:38,380 INFO sqlalchemy.engine.Engine COMMIT
✅ Recreated data/external/cultpass.db with fresh schema


In [4]:
engine = create_engine(f"sqlite:///{cultpass_db}", echo=False)
cultpass.Base.metadata.create_all(engine)

**Experiences**

In [5]:
experience_data = []

with open('data/external/cultpass_experiences.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        experience_data.append(json.loads(line))

In [6]:
experience_data

[{'title': 'Carnival History Tour in Olinda',
  'description': "Discover the origins and vibrant traditions of Pernambuco's Carnival.",
  'location': 'Pernambuco, Brazil'},
 {'title': 'Sunset Paddleboarding',
  'description': 'Glide across calm waters at golden hour with all gear included.',
  'location': 'Santa Catarina, Brazil'},
 {'title': 'Pelourinho Colonial Walk',
  'description': 'Wander through colorful streets and learn about Afro-Brazilian history.',
  'location': 'Bahia, Brazil'},
 {'title': 'Samba Night at Lapa',
  'description': 'Dance the night away at a traditional samba club in the Lapa arches.',
  'location': 'Rio de Janeiro, Brazil'},
 {'title': 'Christ the Redeemer Experience',
  'description': 'Take a guided trip to one of the New Seven Wonders of the World with historical context.',
  'location': 'Rio de Janeiro, Brazil'},
 {'title': 'Modern Art at MASP',
  'description': 'Enjoy a guided visit to the São Paulo Museum of Art with insights into its top collections.',

In [7]:
with get_session(engine) as session:
    experiences = []

    for idx, experience in enumerate(experience_data):
        exp = cultpass.Experience(
            experience_id=str(uuid.uuid4())[:6],
            title=experience["title"],
            description=experience["description"],
            location=experience["location"],
            when=datetime.now() + timedelta(days=idx+1),
            slots_available=random.randint(1,30),
            is_premium=(idx % 2 == 0)
        )
        experiences.append(exp)

    session.add_all(experiences)

**User**

In [12]:
cultpass_users = []

with open('data/external/cultpass_users.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_users.append(json.loads(line))

In [13]:
cultpass_users

[{'id': 'a4ab87',
  'name': 'Alice Kingsley',
  'email': 'alice.kingsley@wonderland.com',
  'is_blocked': True},
 {'id': 'f556c0',
  'name': 'Bob Stone',
  'email': 'bob.stone@granite.com',
  'is_blocked': False},
 {'id': '88382b',
  'name': 'Cathy Bloom',
  'email': 'cathy.bloom@florals.org',
  'is_blocked': False},
 {'id': '888fb2',
  'name': 'David Noir',
  'email': 'david.noir@shadowmail.com',
  'is_blocked': True},
 {'id': 'f1f10d',
  'name': 'Eva Green',
  'email': 'eva.green@ecosoul.net',
  'is_blocked': False},
 {'id': 'e6376d',
  'name': 'Frank Ocean',
  'email': 'frank.ocean@seawaves.io',
  'is_blocked': False}]

In [ ]:
with get_session(engine) as session:
    db_users = []
    for user_info in cultpass_users:
        user = cultpass.User(
            user_id=user_info["id"],
            full_name=user_info["name"],
            email=user_info["email"],
            is_blocked=user_info["is_blocked"],
            created_at=datetime.now()
        )
        db_users.append(user)
    session.add_all(db_users) 

**Subscription**

In [15]:
with get_session(engine) as session:
    subscriptions = []
    for user_info in cultpass_users:
        subscription = cultpass.Subscription(
            subscription_id=str(uuid.uuid4())[:6],
            user_id=user_info["id"],
            status=random.choice(["active", "cancelled"]),
            tier=random.choice(["basic", "premium"]),
            monthly_quota=random.randint(2,10),
            started_at=datetime.now()
        )
        subscriptions.append(subscription)

    session.add_all(subscriptions)

**Reservation**

In [16]:
# Applicable to `cultpass_users[0]` at the moment

with get_session(engine) as session:
    experience_ids = [
        exp.experience_id 
        for exp 
        in session.query(cultpass.Experience).all()
    ]

    reservation1 = cultpass.Reservation(
        reservation_id=str(uuid.uuid4())[:6],
        user_id=cultpass_users[0]["id"],
        experience_id=random.choice(experience_ids),
        status="reserved",
    )

    reservation2 = cultpass.Reservation(
        reservation_id=str(uuid.uuid4())[:6],
        user_id=cultpass_users[0]["id"],
        experience_id=random.choice(experience_ids),
        status="reserved",
    )

    session.add_all([reservation1, reservation2])

In [29]:
with get_session(engine) as session:
    # subscriptions = []
    # for user_info in 
    print(len(cultpass_users)) # :

6


In [37]:
# TODO: Add more data
# Please notice that the reservations were set to first user only 
# If you want to simulate more users later, please create more reservations per user

with get_session(engine) as session:
    experience_ids = [
        exp.experience_id 
        for exp 
        in session.query(cultpass.Experience).all()
    ]

    reservations = []

    for uz in range(1, len(cultpass_users)): 
        for i in range(3):
            reservn = cultpass.Reservation(
                reservation_id = str(uuid.uuid4())[:6],
                user_id=cultpass_users[uz]["id"],
                experience_id=random.choice(experience_ids),
                status="reserved",
            )
            reservations.append(reservn)

    session.add_all(reservations)


# Tests

In [31]:
with get_session(engine) as session:
    users = session.query(cultpass.User).all()
    for user in users:
        print(user)

<User(user_id='a4ab87', email='alice.kingsley@wonderland.com', is_blocked=True)>
<User(user_id='f556c0', email='bob.stone@granite.com', is_blocked=False)>
<User(user_id='88382b', email='cathy.bloom@florals.org', is_blocked=False)>
<User(user_id='888fb2', email='david.noir@shadowmail.com', is_blocked=True)>
<User(user_id='f1f10d', email='eva.green@ecosoul.net', is_blocked=False)>
<User(user_id='e6376d', email='frank.ocean@seawaves.io', is_blocked=False)>


In [32]:
with get_session(engine) as session:
    users = session.query(cultpass.User).all()
    for user in users:
        print(user.subscription)

<Subscription(subscription_id='b58e35', user_id='a4ab87', status='cancelled', tier='basic')>
<Subscription(subscription_id='7869ef', user_id='f556c0', status='active', tier='premium')>
<Subscription(subscription_id='b5bb57', user_id='88382b', status='cancelled', tier='premium')>
<Subscription(subscription_id='8ea147', user_id='888fb2', status='cancelled', tier='basic')>
<Subscription(subscription_id='1b1686', user_id='f1f10d', status='cancelled', tier='premium')>
<Subscription(subscription_id='d60d4f', user_id='e6376d', status='cancelled', tier='basic')>


In [38]:
with get_session(engine) as session:
    Reservations = session.query(cultpass.Reservation).all()
    for experience in Reservations:
        print(experience)

<Reservation(reservation_id='f5cd92', user_id='a4ab87', experience_id='6543c3', status='reserved')>
<Reservation(reservation_id='75231f', user_id='a4ab87', experience_id='6543c3', status='reserved')>
<Reservation(reservation_id='af9039', user_id='a4ab87', experience_id='f5410c', status='reserved')>
<Reservation(reservation_id='868a42', user_id='a4ab87', experience_id='c44871', status='reserved')>
<Reservation(reservation_id='b2eb29', user_id='a4ab87', experience_id='8bd7c9', status='reserved')>
<Reservation(reservation_id='fc158f', user_id='a4ab87', experience_id='3603fa', status='reserved')>
<Reservation(reservation_id='0ef9a7', user_id='a4ab87', experience_id='8bd7c9', status='reserved')>
<Reservation(reservation_id='f03382', user_id='a4ab87', experience_id='8bd7c9', status='reserved')>
<Reservation(reservation_id='dec8a8', user_id='a4ab87', experience_id='3603fa', status='reserved')>
<Reservation(reservation_id='ff79a5', user_id='a4ab87', experience_id='1ca520', status='reserved')>


In [39]:
with get_session(engine) as session:
    experiences = session.query(cultpass.Experience).all()
    for experience in experiences:
        print(experience)

<Experience(experience_id='1ca520', title='Carnival History Tour in Olinda', when='2026-08-17 20:03:17.049362')>
<Experience(experience_id='8bd7c9', title='Sunset Paddleboarding', when='2026-08-18 20:03:17.060371')>
<Experience(experience_id='0c4686', title='Pelourinho Colonial Walk', when='2026-08-19 20:03:17.060441')>
<Experience(experience_id='6543c3', title='Samba Night at Lapa', when='2026-08-20 20:03:17.060489')>
<Experience(experience_id='f5410c', title='Christ the Redeemer Experience', when='2026-08-21 20:03:17.060524')>
<Experience(experience_id='c44871', title='Modern Art at MASP', when='2026-08-22 20:03:17.060559')>
<Experience(experience_id='3603fa', title='Ibirapuera Park Bike Ride', when='2026-08-23 20:03:17.060606')>
